# **Q-03: HÀM CHUYỂN WAV → LOG-MEL SPECTROGRAM**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
import torchaudio
from pathlib import Path
from typing import Union
import gdown

In [ ]:
TARGET_SR = 22050
TARGET_LEN = 110250  # 5 giây × 22050 Hz
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 512
F_MIN = 0
F_MAX = TARGET_SR // 2  # 11025 Hz

In [ ]:
def wav_to_logmel(
    path: Union[str, Path],
    n_mels: int,
    device: str = "cpu"
) -> torch.Tensor:
    """
    Đọc file WAV và chuyển thành Log-Mel Spectrogram.

    Tham số:
        path: Đường dẫn file .wav
        n_mels: Số băng tần Mel (64 hoặc 128)
        device: Thiết bị tính toán ('cpu' hoặc 'cuda')

    Trả về:
        Log-Mel Spectrogram: tensor shape [1, n_mels, 216] (dtype: float32)
    """
      # 1. Đọc file âm thanh
    waveform, sr = torchaudio.load(path)
    waveform = waveform.to(device)

    # 2. Chuyển sang mono (nếu là stereo)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # 3. Resample nếu tần số lấy mẫu khác TARGET_SR (22050)
    if sr != TARGET_SR:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=TARGET_SR).to(device)
        waveform = resampler(waveform)
    # 4. Chuẩn hóa độ dài (cắt hoặc đệm padding về đúng TARGET_LEN = 110250)
    num_samples = waveform.shape[1]
    if num_samples > TARGET_LEN:
        waveform = waveform[:, :TARGET_LEN]
    elif num_samples < TARGET_LEN:
        pad_len = TARGET_LEN - num_samples
        waveform = torch.nn.functional.pad(waveform, (0, pad_len))

    # 5. Khởi tạo bộ biến đổi MelSpectrogram
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=TARGET_SR,
        n_fft=N_FFT,
        win_length=WIN_LENGTH,
        hop_length=HOP_LENGTH,
        f_min=F_MIN,
        f_max=F_MAX,
        n_mels=n_mels
    ).to(device)

    # Tính Mel Spectrogram
    mel_spec = mel_transform(waveform)

    # 6. Chuyển sang thang đo Log-amplitude (dB) để có Log-Mel Spectrogram
    # Sử dụng hằng số nhỏ 1e-5 tránh lỗi log(0)
    log_mel_spec = torch.log(mel_spec + 1e-5)

    # Đảm bảo output có kích thước chính xác là [1, n_mels, 216]
    if log_mel_spec.shape[2] > 216:
        log_mel_spec = log_mel_spec[:, :, :216]
    elif log_mel_spec.shape[2] < 216:
        pad_time = 216 - log_mel_spec.shape[2]
        log_mel_spec = torch.nn.functional.pad(log_mel_spec, (0, pad_time))

    return log_mel_spec.float()

In [ ]:
test_file_path = "/content/1-100039-A-10.wav"

file_id = "1XTVb9fwHYZjTzgUydpkW7QbuhsNKj1Lw"
gdrive_url = f"https://drive.google.com/uc?id={file_id}"

try:
    print("Đang tải file âm thanh mẫu chuẩn từ Google Drive...")
    if os.path.exists(test_file_path):
        os.remove(test_file_path) # Xóa file lỗi cũ nếu có
    gdown.download(gdrive_url, test_file_path, quiet=False)
    print("Tải file hoàn tất!")
except Exception as e:
    print("Lỗi trong quá trình tải tệp tin:", e)

def run_unit_tests():
    print("\n--- BẮT ĐẦU KIỂM THỬ (UNIT TEST) ---")
    try:
        mel_128 = wav_to_logmel(test_file_path, n_mels=128, device="cpu")
        assert mel_128.ndim == 3, f"Lỗi: Số chiều mong muốn là 3, nhận được {mel_128.ndim}"
        assert list(mel_128.shape) == [1, 128, 216], f"Lỗi kích thước: Mong muốn [1, 128, 216], nhận được {list(mel_128.shape)}"
        assert mel_128.dtype == torch.float32, f"Lỗi kiểu dữ liệu: Mong muốn float32, nhận được {mel_128.dtype}"
        print("✓ Test Case 1 (n_mels=128) ĐẠT CHUẨN: Kích thước đạt", list(mel_128.shape))

        mel_64 = wav_to_logmel(test_file_path, n_mels=64, device="cpu")
        assert list(mel_64.shape) == [1, 64, 216], f"Lỗi kích thước: Mong muốn [1, 64, 216], nhận được {list(mel_64.shape)}"
        print("✓ Test Case 2 (n_mels=64) ĐẠT CHUẨN: Kích thước đạt", list(mel_64.shape))

        print("\n=> TẤT CẢ CÁC BÀI KIỂM TRA ĐỀU THÀNH CÔNG!")
    except Exception as e:
        print("BÀI KIỂM TRA THẤT BẠI. Chi tiết lỗi:")
        print(e)

run_unit_tests()

Đang tải file âm thanh mẫu chuẩn từ Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1XTVb9fwHYZjTzgUydpkW7QbuhsNKj1Lw
To: /content/1-100039-A-10.wav
100%|██████████| 441k/441k [00:00<00:00, 91.7MB/s]

Tải file hoàn tất!

--- BẮT ĐẦU KIỂM THỬ (UNIT TEST) ---
✓ Test Case 1 (n_mels=128) ĐẠT CHUẨN: Kích thước đạt [1, 128, 216]
✓ Test Case 2 (n_mels=64) ĐẠT CHUẨN: Kích thước đạt [1, 64, 216]

=> TẤT CẢ CÁC BÀI KIỂM TRA ĐỀU THÀNH CÔNG!
